# Notebook 7 - Effect size preservation
Effect size (Cohen's d against CamCAN) of each disease before and after harmonization, on the 240 synthetic control sites.
The table of |ΔSD| and the figure `effect_size_by_disease` are saved in `RESULTS/EFFECT_SIZE/effect_size_by_disease`.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed

from src.clinical_combat.robust.robust_utils import (
    get_camcan_file,
    get_metrics,
    remove_covariates_effects,
)


In [ ]:
MAIN_FOLDER = "RESULTS/EFFECT_SIZE"
OUTPUT_FOLDER = f"{MAIN_FOLDER}/effect_size_by_disease"

# Reference values (gt_train_*) and values harmonized by each method
GT_ROOT = "DATA/processed/synthetic_sites/classic/harmonized_AUG_5/ALL"
HARM_ROOT = "RESULTS/MAE_TEST/PROCESS/classic2/classic/ALL"

metrics = get_metrics()
diseases = ["AD", "MCI", "TBI", "SCHZ", "BIP", "ADHD"]
disease_ratios = [3, 10, 30, 50, 70, 80]  # percentage of patients in a site
num_tests = 40  # synthetic control sites per ratio, 100 subjects each
sites = [(ratio, idx) for ratio in disease_ratios for idx in range(num_tests)]

# Not white matter bundles, left out of the 45 bundles of CamCAN
DROPPED_BUNDLES = ["left_ventricle", "right_ventricle"]

# Folder of each method in the harmonized files, then its label
# The order is the one of the bars and of the legend
METHODS = {
    "hc": "HC",
    "MLP5_ALL": "MLP",
    "G_MAD": "G_MAD",
    "PCA_MAD": "PCA_MAD",
    "G_ZS": "G_ZS",
    "PCA_ZS": "PCA_ZS",
    "IQR": "IQR",
    "SN": "SN",
    "QN": "QN",
    "MAD": "MAD",
    "ZS": "ZS",
    "VS": "VS",
    "MMS": "MMS",
    "NoRobust": "NO_FILTERING",
}
MIN_PATIENTS = 3  # patients of a disease needed in a site

# Ratios drawn for each disease (at least 10 counted sites each)
SHOWN_RATIOS = {
    **dict.fromkeys(["AD", "MCI", "TBI"], [10, 30, 50, 70, 80]),
    **dict.fromkeys(["SCHZ", "BIP", "ADHD"], [50, 70, 80]),
}

READ_KWARGS = {
    "usecols": [
        "sid",
        "bundle",
        "mean",
        "site",
        "age",
        "sex",
        "handedness",
        "disease",
    ],
    "dtype": {"site": str},
}
N_JOBS = -1


## Effect size functions


In [ ]:
def harm_file(ratio, idx, metric, folder):
    # The hc folder holds the file named NoRobust
    tag = "NoRobust" if folder == "hc" else folder
    return os.path.join(
        HARM_ROOT,
        f"100_{ratio}",
        str(idx),
        metric,
        folder,
        f"ALL_100_patients_{ratio}_percent_{idx}."
        f"{metric}.classic.{tag}.NoRWP.csv",
    )


def stats(df, by):
    """Number, mean and sample SD (ddof = 1) of the values."""
    return df.groupby(by)["mean_no_cov"].agg(n="size", mean="mean", sd="std")


def cohen_d(values, cam):
    """Cohen's d (pooled SD) of each disease against CamCAN, per bundle."""
    # The covariate model is refitted on the site values and CamCAN together
    fit = remove_covariates_effects(
        pd.concat(
            [values.assign(origin="site"), cam.assign(origin="camcan")],
            ignore_index=True,
        )
    )
    patients = fit[(fit["origin"] == "site") & fit["disease"].isin(diseases)]
    g = stats(patients, ["bundle", "disease"]).join(
        stats(fit[fit["origin"] == "camcan"], "bundle"),
        on="bundle",
        rsuffix="_cam",
    )
    pooled_sd = np.sqrt(
        ((g["n"] - 1) * g["sd"] ** 2 + (g["n_cam"] - 1) * g["sd_cam"] ** 2)
        / (g["n"] + g["n_cam"] - 2)
    )
    return pd.DataFrame(
        {"n_patients": g["n"], "d": (g["mean"] - g["mean_cam"]) / pooled_sd}
    )


def site_rows(ratio, idx):
    """d_ref and d_harm of every row of one synthetic control site."""
    out = []
    for metric in metrics:
        cam = pd.read_csv(get_camcan_file(metric, cleaned=True), **READ_KWARGS)
        cam = cam[~cam["bundle"].isin(DROPPED_BUNDLES)]
        ref_file = os.path.join(
            GT_ROOT,
            f"100_{ratio}",
            str(idx),
            f"gt_train_100_{ratio}_{idx}_{metric}.csv",
        )
        d_ref = cohen_d(pd.read_csv(ref_file, **READ_KWARGS), cam)
        for folder, method in METHODS.items():
            harm_df = pd.read_csv(
                harm_file(ratio, idx, metric, folder), **READ_KWARGS
            )
            d_harm = cohen_d(harm_df, cam)
            g = d_ref.rename(columns={"d": "d_ref"}).assign(d_harm=d_harm["d"])
            g = g[g["n_patients"] >= MIN_PATIENTS].reset_index()
            out.append(
                g.assign(
                    ratio=ratio, site_id=idx, metric=metric, method=method
                )
            )
    return pd.concat(out, ignore_index=True)


## Run the calculations


In [ ]:
rows = pd.concat(
    Parallel(n_jobs=N_JOBS)(
        delayed(site_rows)(ratio, idx) for ratio, idx in sites
    ),
    ignore_index=True,
)
print(f"{len(sites)} sites, {len(rows):,} rows")


## Table and figure


In [ ]:
rows["abs_err"] = (rows["d_harm"] - rows["d_ref"]).abs()  # |ΔSD|

# Same rows for every counted site, so the flat mean is the mean of site means
assert (
    rows.groupby(["ratio", "site_id", "disease", "method"]).size().nunique()
    == 1
)

n_sites = (
    rows.groupby(["ratio", "disease"])["site_id"].nunique().rename("n_sites")
)
table = (
    rows.groupby(["ratio", "disease", "method"])["abs_err"]
    .mean()
    .rename("abs_delta_sd")
    .reset_index()
    .join(n_sites, on=["ratio", "disease"])
)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
table_path = os.path.join(OUTPUT_FOLDER, "abs_delta_sd_by_disease.csv")
table.to_csv(table_path, index=False)
print(f"{len(table)} rows written in {table_path}")

# |ΔSD| at 80 % of patients for AD and MCI
at80 = table[
    (table["ratio"] == 80)
    & table["disease"].isin(["AD", "MCI"])
    & table["method"].isin(["HC", "MLP", "NO_FILTERING"])
]
display(
    at80[["disease", "method", "abs_delta_sd", "n_sites"]]
    .round(4)
    .reset_index(drop=True)
)


In [ ]:
order = list(METHODS.values())
# HC, MLP, the 11 other methods in grey from dark to light, NO_FILTERING
grey = plt.cm.gray(np.linspace(0.05, 0.95, len(order) - 3))
colors = dict(zip(order, ["chartreuse", "yellow", *grey, "red"]))
means = table.pivot_table(
    index=["disease", "ratio"], columns="method", values="abs_delta_sd"
)

fig, axes = plt.subplots(2, 3, figsize=(17, 9.5), sharey=True)
for ax, disease in zip(axes.flat, diseases):
    ratios = SHOWN_RATIOS[disease]
    # One group of 14 bars per ratio
    x, width = np.arange(len(ratios)), 0.8 / len(order)
    for k, method in enumerate(order):
        ax.bar(
            x + (k - len(order) / 2 + 0.5) * width,
            means.loc[disease].loc[ratios, method],
            width,
            color=colors[method],
            edgecolor="black",
            linewidth=0.5,
            label=method,
        )
    ax.set_xticks(x, [f"{ratio}%" for ratio in ratios], fontsize=12)
    ax.set_title(disease, fontsize=13)
    ax.set_ylim(0, 0.6)
    ax.tick_params(labelsize=11)
for ax in axes[:, 0]:
    ax.set_ylabel("|Δ SD|", fontsize=13)
for ax in axes[1]:
    ax.set_xlabel("Proportion of patients (%)", fontsize=13)
fig.legend(
    *axes[0, 0].get_legend_handles_labels(),
    loc="center right",
    fontsize=13,
    frameon=False,
)
fig.tight_layout(rect=[0, 0, 0.88, 1])
fig.savefig(os.path.join(OUTPUT_FOLDER, "effect_size_by_disease.png"), dpi=150)
plt.show()
